In [40]:
from dotenv import load_dotenv
load_dotenv()

from langgraph.graph import START,END,StateGraph 
from pydantic import BaseModel
from langchain_groq import ChatGroq
from langgraph.types import interrupt,Command
from langgraph.checkpoint.memory import InMemorySaver

In [41]:
llm=ChatGroq(model="openai/gpt-oss-20b")

In [ ]:
class MailState(BaseModel):
    query:str=""
    draft:str=""
    human_feedback:str=""
    final_response:str=""
    

In [54]:
def draft_email(state:MailState):
    if state.human_feedback:
        draft=llm.invoke(f"""
            Rewrite this draft mail for:{state.query},
            You have generated this mail:{state.draft},
            Human feedback to apply:{state.human_feedback}
         """).content
        
    else :
        draft=llm.invoke(state.query).content
    state.draft=draft
    return state 

def human_feedback(state:MailState):
    feedback=interrupt({
        "draft_email":state.draft,
        "question":"Do you want to continue or re-write the mail"
    })
    fb=(feedback or "").strip().lower()
    if fb in ("approve","ok","accept","yes"):
        state.human_feedback=""
        return state
    else :
        state.human_feedback=feedback
        return state 

def final_node(state:MailState):
    if state.human_feedback:
        state.final_response=state.human_feedback
        return state
    else :
        print("Email send successfully")
        state.final_response="Email send successfully"
        return state

def conditional_routing(state:MailState):
    if state.human_feedback:
        return "draft_email"
    return "final_node" 
    

In [50]:
graph=StateGraph(MailState)
graph.add_node("draft_email",draft_email)
graph.add_node("human_feedback", human_feedback)
graph.add_node("final_node", final_node)

graph.add_edge(START,"draft_email")
graph.add_edge("draft_email","human_feedback")
graph.add_conditional_edges("human_feedback",conditional_routing)
graph.add_edge("final_node",END)


In [51]:
graph=graph.compile(checkpointer=InMemorySaver())
config={"configurable":{"thread_id":"1"}}
res=graph.invoke({"query":"I want to send a email for medical leave,dates will be 15 sept to 17 sept, so create a simple email to ceo"},config=config)

print(res["draft"])

**Subject:** Medical Leave Request – 15 Sept – 17 Sept  

Dear [CEO’s Name],

I hope you are well. I am writing to inform you that I will need to take medical leave from **15 September to 17 September** due to a scheduled medical appointment and subsequent recovery period.  

I will ensure that all urgent tasks are delegated and that my responsibilities are covered during my absence. I will be reachable by phone/email for any critical matters and will return to work on 18 September.

Thank you for your understanding. I will provide the necessary medical documentation upon my return.

Best regards,

[Your Full Name]  
[Your Position]  
[Contact Information]


In [56]:
res=graph.invoke(Command(resume="yes"), config=config)
print(res["draft"])

Email send successfully
**Subject:** Medical Leave Request – 15 Sept – 17 Sept  

Dear [CEO’s Name],

I am writing to request medical leave from **15 September to 17 September**. I will be back in the office on **18 September**.  

During my absence, I have briefed my team on urgent tasks and will remain reachable by phone or e‑mail for any critical matters.

Thank you for your understanding.

Best regards,

Raushan Kumar  
System Engineer  
Phone: 763‑186‑7534
